In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_classic.retrievers import EnsembleRetriever

In [2]:
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
file_name = "data_files/Employee Performance.pdf"
loader = PyPDFLoader(file_name)
docs = loader.load()
print(len(docs[0].page_content))

1184


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)

print(chunks[0].page_content)
print(chunks[0].metadata)
print(len(chunks))

Employee Performance & Operations 
Report 
Quarterly Review – Q1 2026 
Prepared by: Human Resources & Operations Team 
Company: Apex Dynamics Pvt. Ltd. 
1. Executive Summary
{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-05-20T10:22:57+05:30', 'author': 'Gokul', 'moddate': '2026-05-20T10:22:57+05:30', 'source': 'data_files/Employee Performance.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'start_index': 0}
23


In [5]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

In [6]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("vector_db created.")

vector_db created.


In [7]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2

In [8]:
document_ids = vectorstore.add_documents(documents=chunks)

print(document_ids[:3])

['3fd7f3d3-0c9b-427c-84ef-d018132357e0', 'f46abec2-e0e2-481e-83f5-d9c35a9eee3f', '51c17ba4-c6fd-4516-a24b-35480cd431a2']


In [9]:
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [10]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        semantic_retriever
    ],
    weights=[0.4, 0.6]
)

In [11]:
query = "What is Virtusa?"

results = hybrid_retriever.invoke(query)

for result in results:
    print(result.page_content)
    print("-" * 50)

associaon with Virtusa Consulng services private Limited.
 
 
Employee Signature
IP Address: 27.5.170.212
Name:Gokul Krishna Balaji Date of Signing:
14/05/2026
--------------------------------------------------
Document Version: 1.4 
Last Updated: 2026-03-18 
Prepared Using: Internal Operations Reporting Suite v3.2 
Confidentiality Level: Internal Use Only
--------------------------------------------------
Security audit findings: 
• 2 medium-risk vulnerabilities detected  
• 0 critical vulnerabilities detected  
• Patch deployment success rate: 98%  
10. Appendix 
Document Version: 1.4 
Last Updated: 2026-03-18
--------------------------------------------------


In [12]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""

    retrieved_docs = hybrid_retriever.invoke(query)

    serialized = "\n\n".join(
        (
            f"Source: {doc.metadata}\n"
            f"Content: {doc.page_content}"
        )
        for doc in retrieved_docs
    )

    return serialized, retrieved_docs

In [13]:
model = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.7,
    api_key=api_key,
    max_completion_tokens=1000
)

In [14]:
from pydantic import BaseModel

class ResponseFormat(BaseModel):
    message: str

In [15]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

tools = [retrieve_context]

prompt = """
You have access to a tool that retrieves context from a company report. Use the tool to help answer user queries. Respond under 30 words.
"""

agent = create_agent(model=model, tools=tools, response_format=ToolStrategy(ResponseFormat), system_prompt=prompt)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the number of new hires this quarter?"}]}
)

In [17]:
result['structured_response'].message

'There are 17 new hires this quarter.'